In [ ]:
# Databricks notebook source
# MAGIC %md
# MAGIC # 2️⃣ Silver Layer: Window Leads & Agregasi Level
# MAGIC **Arsitektur**: Medallion Pipeline  
# MAGIC **Dataset**: XAUUSD H1 OHLCV  
# MAGIC **Tujuan**: Menghitung window leads (k=1..6), agregasi level D-1 (PDH/PDL) dan W-1 (PWH/PWL), LEFT join level ke candle H1.

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# ==============================================================================
# KONFIGURASI & PARAMETER
# ==============================================================================
INPUT_TABLE = "smt7_research.xauusd.xauusd_liquidity_bronze"
OUTPUT_TABLE = "smt7_research.xauusd.xauusd_liquidity_silver"
N_WINDOW = 6

spark.conf.set("spark.sql.session.timeZone", "UTC")

### 1. Load Data Bronze

In [ ]:
df_bronze = spark.table(INPUT_TABLE)

### 2. Transformasi: Window Leads (k=1..6) Global

In [ ]:
# Window global berdasarkan urutan waktu. Dihitung sebelum join untuk menjamin urutan asli.
window_time = Window.orderBy("timestamp")
df_silver = df_bronze

for k in range(1, N_WINDOW + 1):
    df_silver = df_silver.withColumn(f"close_lead_{k}", F.lead("close", k).over(window_time)) \
                         .withColumn(f"high_lead_{k}",  F.lead("high",  k).over(window_time)) \
                         .withColumn(f"low_lead_{k}",   F.lead("low",   k).over(window_time))

# T3: Simpan timestamp candle ke-N untuk mengukur rentang waktu sesungguhnya
df_silver = df_silver.withColumn("ts_lead_N", F.lead("timestamp", N_WINDOW).over(window_time))

### 3. Agregasi Level H-1 & W-1 (T13, T9)

In [ ]:
# Level diagregasi dari H1, BUKAN tabel D1/W1 broker.
# PDH/PDL: High/Low hari kemarin (D-1)
df_daily_levels = (df_bronze.groupBy("session_date")
    .agg(F.max("high").alias("d_high"), F.min("low").alias("d_low")))

wd = Window.orderBy("session_date")
df_daily_levels = (df_daily_levels
    .withColumn("PDH", F.lag("d_high", 1).over(wd))
    .withColumn("PDL", F.lag("d_low",  1).over(wd))
    .select("session_date", "PDH", "PDL"))

# PWH/PWL: High/Low pekan kemarin (W-1)
df_weekly_levels = (df_bronze.groupBy("week_start")
    .agg(F.max("high").alias("w_high"), F.min("low").alias("w_low")))

ww = Window.orderBy("week_start")
df_weekly_levels = (df_weekly_levels
    .withColumn("PWH", F.lag("w_high", 1).over(ww))
    .withColumn("PWL", F.lag("w_low",  1).over(ww))
    .select("week_start", "PWH", "PWL"))

### 4. LEFT Join Level & Kolom Pendukung Crossover (T6)

In [ ]:
# T6: Gunakan LEFT JOIN agar baris tanpa level (orphan) tetap tercatat di funnel
n_before = df_silver.count()
df_silver = df_silver.join(df_daily_levels, on="session_date", how="left") \
                     .join(df_weekly_levels, on="week_start", how="left")

n_orphan_d = df_silver.filter(F.col("PDH").isNull()).count()
n_orphan_w = df_silver.filter(F.col("PWH").isNull()).count()

print(f"[FUNNEL] Total Baris H1          : {n_before}")
print(f"[FUNNEL] Orphan Level D-1 (PDH/L): {n_orphan_d}")
print(f"[FUNNEL] Orphan Level W-1 (PWH/L): {n_orphan_w}")

# Hitung prev_high / prev_low global untuk deteksi crossover yang presisi
df_silver = df_silver.withColumn("prev_high", F.lag("high", 1).over(window_time)) \
                     .withColumn("prev_low",  F.lag("low",  1).over(window_time))

### 5. Simpan ke Delta Table

In [ ]:
df_silver.write.format("delta").mode("overwrite").saveAsTable(OUTPUT_TABLE)
print(f"Silver layer saved to {OUTPUT_TABLE}.")
display(df_silver.limit(5))